# TP : Introduction à l'apprentissage automatique avec les manchots de Palmer

**Durée estimée :** 2 heures · **Niveau :** premier cycle universitaire, bases de Python et de statistique

## Objectifs

À la fin de ce TP, vous saurez :

1. identifier le **type d'apprentissage** d'un problème et justifier ce choix ;
2. charger un jeu de données public depuis une URL et l'explorer graphiquement ;
3. préparer les données (valeurs manquantes, variables catégorielles, mise à l'échelle) sans fuite de données ;
4. entraîner et comparer plusieurs modèles de classification à partir d'une référence simple ;
5. évaluer un modèle avec une matrice de confusion, un rapport de classification et une courbe d'apprentissage ;
6. interpréter un modèle et en tirer des conclusions.

## 1. De quel type d'apprentissage s'agit-il ?

Le problème est le suivant : **à partir de mesures morphologiques d'un manchot (longueur de bec, profondeur de bec, longueur de nageoire, masse, île, sexe), prédire son espèce** (Adelie, Chinstrap ou Gentoo).

Ce problème relève de l'**apprentissage supervisé**, plus précisément de la **classification multiclasse** :

- **Supervisé** : chaque observation est associée à une étiquette connue (l'espèce). Le modèle apprend à partir de paires (entrée, sortie) déjà annotées.
- **Classification** : la cible $y$ prend un nombre fini de valeurs (trois espèces), et non une valeur réelle. Si la cible était la masse, ce serait une **régression**.
- **Multiclasse** : il y a plus de deux classes. Une classification binaire n'aurait que deux valeurs possibles.

Pour distinguer ce cas de l'apprentissage non supervisé, il suffit de se demander : **la bonne réponse est-elle connue pour chaque exemple de données ?** Ici, oui : l'espèce est renseignée pour les 344 manchots. Un algorithme de partitionnement, lui, ne recevrait pas l'espèce et chercherait seulement des groupes dans les données.

> **Remarque.** Le même jeu de données pourrait servir à de l'apprentissage non supervisé (par exemple un k-means sans regarder l'espèce). Ce qui détermine le type d'apprentissage, c'est donc la présence de la cible et l'objectif poursuivi, pas seulement les données.

## 2. Le jeu de données

Le jeu de données **Palmer Penguins** (Gorman, Williams et Fraser, 2014) décrit 344 manchots observés sur trois îles de l'archipel Palmer, en Antarctique. Il est mis à disposition publiquement par le projet *seaborn-data* sur GitHub.

| Variable | Type | Description |
|---|---|---|
| `species` | catégorielle (cible) | Espèce : Adelie, Chinstrap ou Gentoo |
| `island` | catégorielle | Île d'observation |
| `bill_length_mm` | numérique | Longueur du bec (mm) |
| `bill_depth_mm` | numérique | Profondeur du bec (mm) |
| `flipper_length_mm` | numérique | Longueur de la nageoire (mm) |
| `body_mass_g` | numérique | Masse corporelle (g) |
| `sex` | catégorielle | Sexe (male, female) |

Le chargement se fait directement depuis l'URL, sans téléchargement manuel.

In [ ]:
# Importation des bibliothèques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reproductibilité et style des graphiques
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Chargement du jeu de données depuis une URL publique
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(URL)
print("Dimensions :", df.shape)
df.head()

## 3. Première exploration

Avant toute modélisation, on examine la structure des données : types des colonnes, valeurs manquantes et statistiques descriptives. Ces étapes servent à repérer les problèmes de qualité qui affecteront le modèle.

In [ ]:
# Types des colonnes et nombre de valeurs non renseignées
df.info()
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

In [ ]:
# Statistiques descriptives des variables numériques
df.describe().round(1)

**Lecture.** Quelques observations ont des valeurs manquantes (2 mesures absentes, 11 sexes inconnus). Nous les traiterons à l'étape de préparation. Les unités sont très différentes : la masse est en grammes (plusieurs milliers) alors que les longueurs sont en millimètres (quelques dizaines). Cela justifiera une **mise à l'échelle** avant les modèles sensibles aux distances.

## 4. Visualisation des données

### 4.1 Répartition des classes

Il faut vérifier si les trois espèces sont équilibrées. Un déséquilibre fort rendrait l'exactitude trompeuse.

In [ ]:
# Diagramme en barres du nombre de manchots par espèce
fig, ax = plt.subplots(figsize=(6, 4))
ordre = df["species"].value_counts().index
sns.countplot(data=df, x="species", order=ordre, hue="species", legend=False, ax=ax)
for patch in ax.patches:
    ax.annotate(f"{int(patch.get_height())}",
                (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.set_title("Nombre de manchots par espèce")
ax.set_xlabel("Espèce")
ax.set_ylabel("Effectif")
plt.tight_layout()
plt.show()

**Lecture.** Les classes sont modérément déséquilibrées (152, 124 et 68 observations). Nous utiliserons donc une **validation stratifiée**, qui conserve ces proportions dans chaque pli, et nous examinerons aussi le rappel par classe, pas seulement l'exactitude globale.

### 4.2 Distributions des variables selon l'espèce

Un histogramme par espèce montre si une variable aide à distinguer les classes : plus les distributions sont séparées, plus la variable est informative.

In [ ]:
# Histogrammes superposés des quatre variables numériques, par espèce
variables_num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
titres = ["Longueur du bec (mm)", "Profondeur du bec (mm)",
          "Longueur de la nageoire (mm)", "Masse corporelle (g)"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, var, titre in zip(axes.ravel(), variables_num, titres):
    sns.histplot(data=df, x=var, hue="species", kde=True, alpha=0.45, ax=ax)
    ax.set_title(titre)
    ax.set_xlabel("")
plt.suptitle("Distribution des mesures selon l'espèce", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Lecture.** La **longueur de la nageoire** et la **masse** séparent nettement les Gentoo des deux autres espèces. La **profondeur du bec** distingue surtout les Adelie. Les variables ne sont donc pas toutes également utiles : c'est une information qu'un modèle apprendra automatiquement, mais qu'il est bon de vérifier.

### 4.3 Relation entre deux variables

Un nuage de points colorié par espèce permet de visualiser une **frontière de séparation** possible dans un plan à deux dimensions.

In [ ]:
# Nuage de points : longueur du bec en fonction de la longueur de la nageoire
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="flipper_length_mm", y="bill_length_mm",
                hue="species", style="species", s=60, alpha=0.85, ax=ax)
ax.set_title("Longueur du bec en fonction de la longueur de la nageoire")
ax.set_xlabel("Longueur de la nageoire (mm)")
ax.set_ylabel("Longueur du bec (mm)")
ax.legend(title="Espèce")
plt.tight_layout()
plt.show()

**Lecture.** Les trois nuages forment des groupes presque distincts. Un modèle linéaire pourrait déjà tracer des frontières raisonnables, ce qui en fera une bonne référence.

### 4.4 Boîtes à moustaches et corrélations

Les boîtes à moustaches résument la médiane, les quartiles et les valeurs extrêmes. La matrice de corrélation indique les liens linéaires entre variables numériques : deux variables très corrélées apportent une information redondante.

In [ ]:
# Boîtes à moustaches par espèce
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
for ax, var, titre in zip(axes, variables_num, titres):
    sns.boxplot(data=df, x="species", y=var, hue="species", legend=False, ax=ax)
    ax.set_title(titre)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation des variables numériques
fig, ax = plt.subplots(figsize=(6.5, 5))
corr = df[variables_num].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Corrélations entre variables numériques")
plt.tight_layout()
plt.show()

**Lecture.** La longueur de la nageoire et la masse sont fortement corrélées : elles mesurent en partie la même chose (la taille de l'animal). Ce n'est pas un problème pour un arbre ou une forêt, mais une régression logistique gagnera à être régularisée.

## 5. Préparation des données

Règles à respecter :

- **Ne jamais utiliser le jeu de test pendant la préparation.** Toute transformation (moyenne, écart-type, catégories) est apprise sur l'entraînement uniquement.
- **Supprimer ou imputer les valeurs manquantes.** Ici, les lignes incomplètes sont peu nombreuses (11 sur 344, soit environ 3 %) ; on les retire, ce qui laisse 333 observations.
- **Encoder les variables catégorielles.** Un modèle ne lit pas le texte : `island` et `sex` deviennent des colonnes 0/1 (*one-hot encoding*).
- **Standardiser les variables numériques** : $z = \dfrac{x - \bar x}{s}$, moyenne nulle et écart-type 1.

Ces étapes sont regroupées dans un `Pipeline` : la même chaîne est alors appliquée à l'entraînement, à la validation et au test.

In [ ]:
# 1) Suppression des lignes incomplètes
df_propre = df.dropna().reset_index(drop=True)
print("Observations après nettoyage :", len(df_propre))

# 2) Séparation entrées / cible
X = df_propre.drop(columns="species")
y = df_propre["species"]

# 3) Découpage stratifié : 75 % entraînement, 25 % test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Entraînement :", X_train.shape[0], "| Test :", X_test.shape[0])
print("Proportions dans l'entraînement :")
print(y_train.value_counts(normalize=True).round(3))

# 4) Prétraitement : standardisation des numériques, encodage des catégorielles
variables_cat = ["island", "sex"]
preprocesseur = ColumnTransformer([
    ("num", StandardScaler(), variables_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), variables_cat),
])

## 6. Entraînement et comparaison de modèles

### 6.1 Principe de la comparaison

Nous comparons plusieurs algorithmes, du plus simple au plus complexe. La **référence** (*baseline*) est un classifieur qui ignore les données et prédit toujours la classe majoritaire. Tout modèle utile doit faire mieux que cette référence.

Pour comparer équitablement, on utilise une **validation croisée à 5 plis sur l'ensemble d'entraînement**. Le jeu de test reste de côté jusqu'à l'évaluation finale.

In [ ]:
# Modèles candidats, chacun précédé du même prétraitement
modeles = {
    "Référence (classe majoritaire)": DummyClassifier(strategy="most_frequent"),
    "Régression logistique": LogisticRegression(max_iter=1000),
    "K plus proches voisins (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=200, random_state=42),
}

pipelines = {nom: Pipeline([("prep", preprocesseur), ("modele", m)])
             for nom, m in modeles.items()}

validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats = []
for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=validation, scoring="accuracy")
    resultats.append({"Modèle": nom, "Exactitude moyenne": scores.mean(), "Écart-type": scores.std()})

tableau = pd.DataFrame(resultats).sort_values("Exactitude moyenne", ascending=False)
tableau.round(3)

In [ ]:
# Graphique : exactitude moyenne en validation croisée, avec barres d'erreur
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(tableau["Modèle"], tableau["Exactitude moyenne"],
        xerr=tableau["Écart-type"], capsize=5, color=sns.color_palette("Set2", len(tableau)))
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.axvline(tableau.loc[tableau["Modèle"].str.startswith("Référence"), "Exactitude moyenne"].values[0],
           color="grey", linestyle="--", label="niveau de la référence")
for i, val in enumerate(tableau["Exactitude moyenne"]):
    ax.text(val + 0.01, i, f"{val:.3f}", va="center")
ax.set_xlabel("Exactitude (validation croisée à 5 plis)")
ax.set_title("Comparaison des modèles")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Lecture.** La référence donne environ 0,44 (elle prédit toujours l'espèce majoritaire). Les modèles appris atteignent des exactitudes bien plus élevées. Les écarts-types indiquent la stabilité : un modèle très bon mais très variable d'un pli à l'autre est moins fiable qu'un modèle un peu moins bon mais stable.

## 7. Évaluation finale sur le jeu de test

Le modèle choisi est entraîné sur tout l'ensemble d'entraînement, puis évalué **une seule fois** sur le jeu de test. Ce score est une estimation de la performance sur de nouveaux manchots.

Nous sélectionnons automatiquement le modèle ayant la meilleure exactitude en validation croisée.

In [ ]:
# Sélection du meilleur modèle selon la validation croisée
meilleur_nom = tableau.iloc[0]["Modèle"]
meilleur_pipe = pipelines[meilleur_nom]
meilleur_pipe.fit(X_train, y_train)
y_pred = meilleur_pipe.predict(X_test)

print("Meilleur modèle :", meilleur_nom)
print(f"Exactitude sur le jeu de test : {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred))

### 7.1 Matrice de confusion

La matrice de confusion croise l'espèce réelle et l'espèce prédite. La diagonale contient les réussites ; les autres cases montrent **quelles espèces sont confondues**, ce que l'exactitude seule ne révèle pas.

In [ ]:
# Matrice de confusion : valeurs absolues et proportions par ligne
classes = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=classes)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes,
            yticklabels=classes, cbar=False, ax=axes[0])
axes[0].set_title("Effectifs")
axes[0].set_xlabel("Espèce prédite")
axes[0].set_ylabel("Espèce réelle")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", xticklabels=classes,
            yticklabels=classes, vmin=0, vmax=1, cbar=False, ax=axes[1])
axes[1].set_title("Proportions par espèce réelle (rappel)")
axes[1].set_xlabel("Espèce prédite")
axes[1].set_ylabel("Espèce réelle")
plt.tight_layout()
plt.show()

**Lecture.** Sur les 84 manchots du test, le modèle n'a commis qu'**une seule erreur** : un Adelie classé comme Chinstrap, deux espèces aux mesures proches. La diagonale de la deuxième figure donne le **rappel** de chaque espèce : la part des manchots de cette espèce correctement retrouvés. Une erreur isolée ne suffit pas à conclure : c'est pourquoi la validation croisée sert à juger la stabilité.

### 7.2 Courbe d'apprentissage

La courbe d'apprentissage répond à une question pratique : **le modèle manque-t-il de données, ou de flexibilité ?**

- Si les courbes d'entraînement et de validation convergent vers un score faible : **sous-apprentissage** (biais élevé).
- Si l'écart reste grand : **surapprentissage** (variance élevée) ; plus de données ou une régularisation aideraient.

In [ ]:
# Courbe d'apprentissage du meilleur modèle
tailles, scores_train, scores_val = learning_curve(
    meilleur_pipe, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=validation, scoring="accuracy", n_jobs=-1
)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
for scores, label, couleur in [(scores_train, "Entraînement", "tab:blue"),
                               (scores_val, "Validation", "tab:orange")]:
    moyenne, ecart = scores.mean(axis=1), scores.std(axis=1)
    ax.plot(tailles, moyenne, "o-", color=couleur, label=label)
    ax.fill_between(tailles, moyenne - ecart, moyenne + ecart, alpha=0.2, color=couleur)
ax.set_xlabel("Nombre d'observations d'entraînement")
ax.set_ylabel("Exactitude")
ax.set_title(f"Courbe d'apprentissage : {meilleur_nom}")
ax.set_ylim(0.5, 1.02)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Lecture.** Avec peu de données, le modèle mémorise l'entraînement (score élevé) mais généralise mal. Quand la taille augmente, les deux courbes se rejoignent : le modèle généralise. Un écart qui se referme indique que l'ajout de données a encore un effet ; une convergence précoce à un niveau élevé indique un modèle suffisant.

## 8. Interprétation du modèle

### 8.1 Importance des variables

La forêt aléatoire fournit une mesure d'importance : la réduction moyenne de l'impureté de Gini apportée par chaque variable. Elle indique quelles mesures pèsent le plus dans la décision. Nous l'entraînons ici sur le même prétraitement pour interpréter le modèle.

In [ ]:
# Importance des variables pour la forêt aléatoire
foret = Pipeline([("prep", preprocesseur),
                  ("modele", RandomForestClassifier(n_estimators=200, random_state=42))])
foret.fit(X_train, y_train)

noms = foret.named_steps["prep"].get_feature_names_out()
noms = [n.replace("num__", "").replace("cat__", "") for n in noms]
importances = pd.Series(foret.named_steps["modele"].feature_importances_, index=noms)
importances = importances.sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot.barh(ax=ax, color="teal")
for i, val in enumerate(importances):
    ax.text(val + 0.005, i, f"{val:.3f}", va="center")
ax.set_xlabel("Importance (réduction moyenne de l'impureté de Gini)")
ax.set_title("Importance des variables, forêt aléatoire")
plt.tight_layout()
plt.show()

**Lecture.** La longueur du bec et celle de la nageoire dominent nettement (environ 60 % de l'importance à elles deux), suivies de la profondeur du bec. L'île et le sexe pèsent peu. Attention toutefois : cette importance dépend du modèle et peut être biaisée quand des variables sont corrélées entre elles, comme nous l'avons vu avec la nageoire et la masse. Elle sert à orienter l'analyse, pas à établir une causalité.

### 8.2 Frontières de décision

Pour visualiser comment un modèle sépare les classes, on se limite à **deux variables** et on colore chaque point du plan selon la prédiction. C'est une vue pédagogique : le modèle complet utilise toutes les variables.

In [ ]:
# Frontières de décision avec deux variables : nageoire et bec
deux = ["flipper_length_mm", "bill_length_mm"]
modele_2d = Pipeline([("scaler", StandardScaler()),
                      ("knn", KNeighborsClassifier(n_neighbors=5))])
modele_2d.fit(X_train[deux], y_train)

x_min, x_max = X_train[deux[0]].min() - 5, X_train[deux[0]].max() + 5
y_min, y_max = X_train[deux[1]].min() - 5, X_train[deux[1]].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grille = pd.DataFrame({deux[0]: xx.ravel(), deux[1]: yy.ravel()})
zz = pd.Series(modele_2d.predict(grille)).map({c: i for i, c in enumerate(classes)}).values.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, zz, levels=len(classes), alpha=0.2, cmap="Set2")
for c in classes:
    masque = y_test == c
    ax.scatter(X_test.loc[masque, deux[0]], X_test.loc[masque, deux[1]],
               label=c, s=60, edgecolor="k", zorder=3)
ax.set_xlabel("Longueur de la nageoire (mm)")
ax.set_ylabel("Longueur du bec (mm)")
ax.set_title("Frontières de décision (KNN, deux variables) et points de test")
ax.legend(title="Espèce réelle")
plt.tight_layout()
plt.show()

**Lecture.** Chaque zone colorée correspond à une espèce prédite. Les points situés près de la limite entre deux zones sont les plus difficiles à classer. Cette figure illustre le principe de l'apprentissage supervisé : le modèle apprend une fonction qui partage l'espace des entrées en régions associées aux étiquettes.

## 9. Prédire une nouvelle observation

Une fois le modèle validé, il sert à prédire l'espèce d'un nouveau manchot. Le pipeline applique automatiquement la même standardisation et le même encodage qu'à l'entraînement.

In [ ]:
# Nouvelle observation fictive, avec ses probabilités par espèce
nouveau = pd.DataFrame([{
    "bill_length_mm": 46.0, "bill_depth_mm": 15.0,
    "flipper_length_mm": 215.0, "body_mass_g": 4800.0,
    "island": "Biscoe", "sex": "female"
}])

print("Espèce prédite :", meilleur_pipe.predict(nouveau)[0])
if hasattr(meilleur_pipe, "predict_proba"):
    probas = pd.Series(meilleur_pipe.predict_proba(nouveau)[0], index=meilleur_pipe.classes_)
    print("\nProbabilités estimées :")
    print(probas.round(3))

## 10. Synthèse et questions

### Ce qu'il faut retenir

1. **Type d'apprentissage** : supervisé, classification multiclasse. La cible (espèce) est connue pour chaque observation.
2. **Démarche** : explorer, nettoyer, prétraiter dans un pipeline, comparer à une référence, valider par validation croisée, évaluer une seule fois sur le test.
3. **Évaluation** : l'exactitude ne suffit pas. La matrice de confusion et le rappel par classe montrent où le modèle se trompe.
4. **Diagnostic** : la courbe d'apprentissage indique si le problème vient du manque de données ou de la complexité du modèle.

### Questions

1. Pourquoi la référence « classe majoritaire » est-elle indispensable ? Quelle exactitude obtiendriez-vous si vous connaissiez seulement l'effectif de chaque espèce ?
2. Modifiez `max_depth` de l'arbre de décision (1, 2, 4, 8, 16). Observez-vous du sous-apprentissage ou du surapprentissage ? Tracez l'exactitude en validation et en entraînement.
3. Retirez la variable `island` et réentraînez la régression logistique. Que se passe-t-il ?
4. Remplacez le découpage aléatoire par un découpage sans stratification (`stratify=None`). Le score change-t-il ? Pourquoi est-ce risqué avec des classes de tailles différentes ?
5. Ce problème pourrait-il être traité en apprentissage non supervisé ? Décrivez une expérience que vous conduiriez, et ce que vous compareriez à l'espèce réelle (indice ARI).

> **Pour aller plus loin.** Remplacez la cible par `body_mass_g` (valeur réelle) : vous passeriez à un problème de **régression**. Quelles métriques (MSE, RMSE, R²) utiliseriez-vous ?